# 02 · From m/z peaks to lipid names

Following the tutorial notebook [`02_annotation_student.ipynb`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/blob/main/notebooks/level1/02_annotation_student.ipynb).

**Kernel:** `cajal-lipidomics`

**What I'm doing here**
- Adducts, de-ionization, and the ppm formula (5 ppm window)
- Match peaks against the LC-MS reference, look at isobaric ties (LPC 15:1 vs LPE 18:1)
- LIPID MAPS as the database layer, parse lipid names into properties
- Annotate every ion and save `02_annotated.h5ad`

**My notes / what surprised me**
- ...

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------

# My own little package. `paths` knows where the tutorial clone, its provided data,
# and my own outputs live, so I never type "../../data/..." by hand.
from lipid_project import paths

paths.ensure_dirs()   # creates data/derived/, figures/, results/ if they're missing (safe to re-run)
paths.summary()       # prints where everything points + which stages I've built so far.
                      # If a line starts with "--", that thing is missing: fix it before going on.

# The usual scientific stack
import numpy as np                 # arrays and math
import pandas as pd                # tables (obs / var are pandas DataFrames)
import matplotlib.pyplot as plt    # plotting
import anndata as ad               # the AnnData container: X (pixels x lipids) + obs + var + obsm + layers

# The tutorial's helper package (installed from my clone in external/).
# Rule from the course: before calling any cl.<module>.<function>, open
# external/lipidomics_tutorial_cajalcourse/src/cajal_lipidomics/<module>.py and read it.
import cajal_lipidomics as cl
cl.style.set_style()               # the course's figure defaults, so plots look clean and consistent

# If this prints without a red error, the kernel is right. A ModuleNotFoundError here almost
# always means the wrong kernel: re-pick "cajal-lipidomics" in the top-right kernel picker.
print("ready. cajal_lipidomics", cl.__version__)

## load

In [ ]:
# ---- load the previous stage --------------------------------------------------------------
# My NB01 output: every tissue pixel x every raw ion. The columns are still named by
# formula + adduct (e.g. "C45H91N2O6P__+H"), not by lipid. Giving them names is this notebook's job.
raw = ad.read_h5ad(paths.stage("raw"))

# Where the annotation references live (from the data bundle):
#   - the in-house LC-MS reference (lipid, adduct, ionized m/z) I match peaks against
#   - QuantitativeLCMS.csv, bulk brain abundances, used to break isobaric ties
#   - structures.sdf, the LIPID MAPS database (every known lipid structure + its neutral mass)
REFS = paths.REFS

print("raw substrate:", raw.shape, "(pixels x ions)")   # expect about 189,011 x 104
print("conditions:", dict(raw.obs["Condition"].value_counts()))

# ---- output -----------------------------------------------------------------------------------
# Same matrix, but var now carries a "lipid" name (or an ion_<mz> placeholder) for every ion:
#   raw.write_h5ad(paths.stage("annotated"))     ->  data/derived/02_annotated.h5ad

## my work

In [ ]:
# my code from here on, one small step per cell, and look at the output each time